# <center> <img src="figs/LogoUFSCar.jpg" alt="Logo UFSCar" width="110" align="left"/>  <br/> <center>Universidade Federal de São Carlos (UFSCar)<br/><font size="4"> Departamento de Computação, campus Sorocaba</center></font>
</p>

<font size="4"><center><b>Disciplina: Processamento de Linguagem Natural</b></center></font>

<font size="3"><center>Prof. Dr. Tiago A. Almeida</center></font>

## <center>Projeto Final - Classificação Automática de Documentos Jurídicos</center>

**Nome**: Anderson Cristiano Sassaki Gonçalves

**RA**: 821675

**Nome**: Lorenzo Grippo Chiachio

**RA**: 823917


---
## Visão geral da solução

O objetivo deste notebook é construir uma solução completa para a competição de classificação de documentos jurídicos da base VICTOR/STF. Cada amostra possui um identificador (`Id`), um texto extraído de documento jurídico (`Body`) e, no conjunto de treino, uma categoria (`Category`). As classes válidas são:

- `0`: Acórdão;
- `1`: Agravo de Recurso Extraordinário (ARE);
- `2`: Despacho;
- `3`: Recurso Extraordinário (RE);
- `4`: Sentença.

A métrica de referência da competição é o F1-Score, portanto os experimentos priorizam `f1_macro`, que penaliza modelos que acertam apenas a classe majoritária. A organização segue o padrão usado no projeto anterior de Aprendizado de Máquina: o notebook documenta as decisões e chama funções implementadas em `scripts/`, enquanto cada etapa do pipeline fica isolada o suficiente para ser refeita sem quebrar as demais.

O fluxo completo executado aqui é:

1. Carregamento e validação dos CSVs da competição;
2. Análise exploratória dos textos e das classes;
3. Pré-processamento textual robusto para ruídos de OCR e linguagem jurídica;
4. Extração de sinais linguísticos e jurídicos auxiliares;
5. Modelos clássicos com TF-IDF;
6. Validação cruzada dos melhores modelos clássicos;
7. Embeddings densos com SentenceTransformers;
8. Fine-tuning de Transformer no melhor dispositivo disponível (`cuda`, `mps` ou `cpu`);
9. Geração de arquivos de submissão.


---
## Dependências e estratégia de dispositivo

A célula abaixo instala apenas o que estiver ausente. A instalação é feita dentro do ambiente Python usado pelo notebook, o que evita depender de comandos externos ou de configuração manual. O conjunto `requirements.txt` cobre análise, visualização e modelos clássicos. O conjunto `requirements-advanced.txt` cobre embeddings e Transformers. A dependência `bitsandbytes`, necessária apenas para QLoRA em GPUs NVIDIA/Linux, ficou separada em `requirements-nvidia.txt` para não quebrar máquinas Mac/MPS.

Depois das dependências, o notebook detecta automaticamente o dispositivo disponível:

- se houver GPU NVIDIA, usa `cuda`;
- se estiver em Apple Silicon com suporte PyTorch, usa `mps`;
- se nenhum acelerador estiver disponível, usa `cpu`.

Não há flags manuais para ativar ou desativar partes do pipeline. O mesmo fluxo é executado sempre; o que muda é o dispositivo, o tamanho de batch e a precisão usada pelo treino Transformer.


In [1]:
import importlib.util
import subprocess
import sys

BASE_MODULES = ["pandas", "numpy", "sklearn", "matplotlib", "seaborn", "joblib"]
ADVANCED_MODULES = ["torch", "transformers", "datasets", "sentence_transformers", "accelerate"]

missing_base = [pkg for pkg in BASE_MODULES if importlib.util.find_spec(pkg) is None]
if missing_base:
    print("Instalando dependências base ausentes:", missing_base)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
else:
    print("Dependências base já disponíveis.")

missing_advanced = [pkg for pkg in ADVANCED_MODULES if importlib.util.find_spec(pkg) is None]
if missing_advanced:
    print("Instalando dependências avançadas ausentes:", missing_advanced)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-advanced.txt"])
else:
    print("Dependências avançadas já disponíveis.")

importlib.invalidate_caches()
print("Ambiente pronto para executar o pipeline completo.")


---
## Importação dos módulos do projeto

Nesta etapa, importamos apenas as funções públicas dos scripts. Essa separação é importante porque permite que o notebook fique enxuto e explicativo, enquanto a lógica que pode ser testada, reaproveitada ou substituída fica nos arquivos `.py` correspondentes.

Os quatro scripts principais são:

- `analise_exploratoria.py`: leitura dos dados, validações, estatísticas e gráficos;
- `preprocessamento.py`: limpeza textual, normalização jurídica, chunking e features linguísticas;
- `experimentos.py`: modelos clássicos, embeddings, Transformers e submissões;
- `analise_resultados.py`: rankings, matrizes de confusão, relatórios e tabelas finais.


In [2]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
MODEL_DIR = PROJECT_DIR / "models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str((PROJECT_DIR / "scripts").resolve()))

from analise_exploratoria import (
    add_text_statistics,
    get_top_terms,
    load_competition_data,
    plot_class_distribution,
    plot_text_length_distribution,
    plot_top_terms,
    summarize_dataset,
    validate_schema,
    split_labeled_unlabeled,
)
from preprocessamento import (
    CLASS_NAMES,
    TextPreprocessConfig,
    add_clean_text_column,
    extract_basic_nlp_features,
    extract_legal_signal_features,
)
from experimentos import (
    ExperimentConfig,
    auto_runtime_profile,
    build_embedding_classifier_zoo,
    build_sentence_embedding_matrix,
    build_chunked_sentence_embedding_matrix,
    build_sparse_model_zoo,

    build_training_set_with_pseudo_labels,
    cross_validate_best_models,

    detect_label_issues_with_cv,
    evaluate_embedding_classifier,
    evaluate_model_zoo,
    fine_tune_transformer_classifier,
    fit_embedding_classifier_on_full_train,
    fit_model_on_full_train,
    generate_embedding_submission,
    generate_submission,
    generate_transformer_submission,

    pseudo_label_unlabeled_samples,
    save_model,
    select_best_model,
    transformer_candidate_models,
)
from analise_resultados import (
    classification_report_frame,
    plot_confusion_matrix,
    plot_metric_comparison,
    rank_results,
    save_results_table,
)

pd.set_option("display.max_colwidth", 160)

runtime_profile = auto_runtime_profile(prefer_transformer=True)
print("Diretório do projeto:", PROJECT_DIR)
print("Perfil automático:", runtime_profile["notes"])
display(pd.DataFrame([
    {
        "device": runtime_profile["device"],
        "advanced_available": runtime_profile["advanced_available"],
        "classical_model_zoo": runtime_profile["classical_model_zoo"],
        "run_classical_baselines": runtime_profile["run_classical_baselines"],
        "run_embeddings": runtime_profile["run_embeddings"],
        "run_transformer": runtime_profile["run_transformer"],
        "batch_transformer": runtime_profile["transformer_batch_size"],
        "max_length_transformer": runtime_profile["transformer_max_length"],
    }
]))


---

## 1. Carregamento e validação dos dados

Os arquivos esperados são `train.csv`, `test.csv` e `sample_submission.csv`. O carregador procura primeiro em `data/`, mas também aceita os arquivos na raiz do projeto, o que facilita alternar entre execução local, Kaggle e servidor remoto.

Um detalhe importante observado nesta base é a presença de amostras com `Category = -1`. Essas linhas não são rótulos válidos da competição, então são separadas do conjunto supervisionado em `train_unlabeled`. Elas não são descartadas: mais à frente, o notebook usa um modelo probabilístico de alta confiança para propor pseudo-rótulos e reaproveitar parte dessas amostras no treino final.

Também tratamos a possibilidade de rótulos incorretos. Depois dos baselines clássicos, uma etapa de auditoria com previsões fora da dobra (`out-of-fold`) sinaliza exemplos em que o rótulo original discorda de uma previsão muito confiante. Esses casos são salvos para inspeção e apenas correções extremamente confiantes entram automaticamente no conjunto final.


In [3]:
train_df, test_df, sample_df = load_competition_data(DATA_DIR)
schema_report = validate_schema(train_df, test_df, sample_df)
train_labeled, train_unlabeled = split_labeled_unlabeled(train_df)

print(f"Total no train.csv: {len(train_df)}")
print(f"Amostras rotuladas válidas: {len(train_labeled)}")
print(f"Amostras sem rótulo válido (Category=-1 ou fora de 0..4): {len(train_unlabeled)}")
print(f"Amostras de teste: {len(test_df)}")

print("Classes usadas no treinamento:")
for label, name in CLASS_NAMES.items():
    print(f"{label}: {name}")

schema_report


In [4]:
display(train_labeled.head())
display(test_df.head())
if sample_df is not None:
    display(sample_df.head())


---
## 2. Análise exploratória

A análise exploratória tem três objetivos principais:

1. Entender o desbalanceamento entre as cinco classes;
2. Medir o tamanho e a variabilidade dos documentos;
3. Identificar termos e expressões que ajudam a caracterizar cada tipo de peça jurídica.

Esse diagnóstico orienta as escolhas seguintes. Por exemplo, se uma classe é muito menor que as demais, modelos com `class_weight='balanced'` e avaliação por `f1_macro` se tornam mais adequados que acurácia pura. Se os textos são muito longos, também precisamos pensar em chunking ou modelos de contexto longo.


In [5]:
eda = summarize_dataset(train_labeled)
train_stats = eda["data_with_stats"]

print("Distribuição de classes")
display(eda["class_distribution"])

print("Valores ausentes")
display(eda["missing_summary"])

print("Resumo de tamanho por classe")
display(eda["length_summary"])


In [6]:
plot_class_distribution(eda["class_distribution"])
plot_text_length_distribution(train_stats, metric="n_words")


A distribuição de classes e de tamanho dos textos deve ser lida junto com o contexto jurídico. Classes como RE e ARE tendem a compartilhar vocabulário, o que pode causar confusão entre elas. Despachos, por outro lado, podem conter textos mais curtos e fórmulas administrativas recorrentes. A análise de termos por classe abaixo ajuda a verificar se o modelo terá pistas lexicais claras ou se dependerá de representações mais semânticas.


In [7]:
top_terms = get_top_terms(train_labeled, n_terms=12)
display(top_terms.head(30))
plot_top_terms(top_terms, n_terms=10)


---
## 3. Pré-processamento textual

Os textos da base vêm de documentos jurídicos reais e podem carregar problemas típicos de OCR: quebras de linha artificiais, hifenização no meio de palavras, assinaturas digitais, carimbos, números processuais e referências legais em formatos variados.

O pré-processamento adotado aqui é propositalmente modular. A função `add_clean_text_column` cria uma coluna `Body_clean`, que passa a ser a interface entre limpeza e modelagem. Se no futuro a limpeza for trocada por uma abordagem melhor, basta preservar essa coluna final para que os modelos continuem funcionando.

As decisões principais são:

- converter para minúsculas;
- remover acentos para reduzir variações superficiais;
- normalizar números de processo, datas, valores monetários e referências a artigos de lei;
- manter stopwords por padrão, pois em textos jurídicos termos funcionais e fórmulas podem carregar sinal de estilo e tipo documental;
- remover tokens muito curtos e ruídos evidentes.


In [8]:
preprocess_config = TextPreprocessConfig(
    lowercase=True,
    strip_accents=True,
    normalize_legal_refs=True,
    remove_urls_emails=True,
    remove_stopwords=False,
    min_token_len=2,
    keep_digits=False,
)

train_clean = add_clean_text_column(train_labeled, config=preprocess_config)
test_clean = add_clean_text_column(test_df, config=preprocess_config)

train_clean[["Id", "Category", "Body", "Body_clean"]].head(3)


In [9]:
train_clean_stats = add_text_statistics(train_clean, text_col="Body_clean")
plot_text_length_distribution(train_clean_stats, metric="n_words")


---
## 4. Tarefas básicas de PLN e features de domínio

O enunciado do projeto pede o uso de tarefas básicas de PLN quando fizer sentido. Para este problema, usamos duas famílias de atributos auxiliares:

1. **Sinais jurídicos determinísticos**, como frequência de expressões associadas a acórdão, ARE, RE, despacho e sentença;
2. **Sinais linguísticos opcionais**, como contagens de PoS e entidades nomeadas via spaCy, quando o modelo de português estiver disponível.

Essas features não substituem TF-IDF nem Transformers. Elas funcionam como uma camada interpretável para análise: ajudam a entender que pistas jurídicas o pipeline consegue observar diretamente.


In [10]:
legal_features = extract_legal_signal_features(train_clean, text_col="Body_clean")
nlp_features = extract_basic_nlp_features(train_clean, text_col="Body")

print("Features jurídicas:", legal_features.shape)
display(legal_features.head())

print("Features básicas de PLN/fallback:", nlp_features.shape)
display(nlp_features.head())


---
## 5. Modelos clássicos com TF-IDF

Os modelos clássicos são a base de comparação. Eles têm duas vantagens importantes neste projeto:

- são fortes em classificação textual quando há vocabulário discriminativo;
- são mais baratos que Transformers, o que permite testar rapidamente hipóteses de limpeza, n-gramas e balanceamento.

A bateria inclui modelos complementares:

- `DummyClassifier`, para estabelecer o mínimo aceitável;
- `ComplementNB`, muito competitivo em texto esparso e desbalanceado;
- Regressão Logística com TF-IDF de palavras;
- LinearSVC com n-gramas de caracteres, robusto a ruídos de OCR;
- SGD com perda logística;
- união de TF-IDF de palavras e caracteres.

A métrica principal é `f1_macro`, pois ela dá peso semelhante para classes frequentes e raras.


In [11]:
experiment_config = ExperimentConfig(
    text_col="Body_clean",
    target_col="Category",
    id_col="Id",
    random_state=42,
    validation_size=0.2,
    cv_folds=5,
    scoring="f1_macro",
    output_dir=str(OUTPUT_DIR),
)

classical_models = build_sparse_model_zoo(random_state=experiment_config.random_state)
results_df, fitted_models = evaluate_model_zoo(
    train_clean,
    models=classical_models,
    config=experiment_config,
)

ranked = rank_results(results_df, metric="f1_macro")
display(ranked)
plot_metric_comparison(results_df, metric="f1_macro")

results_path = save_results_table(results_df, OUTPUT_DIR / "model_results_holdout.csv")
print("Resultados holdout salvos em:", results_path)


---
## 6. Validação cruzada dos melhores modelos clássicos

O holdout é rápido e útil para comparar muitas alternativas, mas uma única divisão pode favorecer ou prejudicar modelos por acaso. Por isso, após o ranking inicial, aplicamos validação cruzada apenas nos três melhores modelos clássicos. Essa etapa é mais custosa, porém fornece uma estimativa mais estável do desempenho médio.


In [12]:
top_model_names = ranked.head(3)["model"].tolist()
cv_results = cross_validate_best_models(
    train_clean,
    model_names=top_model_names,
    models=build_sparse_model_zoo(random_state=experiment_config.random_state),
    config=experiment_config,
)

cv_results_path = OUTPUT_DIR / "model_results_cv_top3.csv"
cv_results.to_csv(cv_results_path, index=False)

display(cv_results)
print("Resultados de validação cruzada salvos em:", cv_results_path)


---
## 7. Análise do melhor modelo clássico

Nesta etapa, olhamos para além do número agregado. A matriz de confusão mostra quais classes estão sendo trocadas entre si, e o relatório por classe mostra se o problema está em precisão, revocação ou ambos.

Essa leitura é fundamental para orientar melhorias futuras. Se a classe `2` tem baixo recall, por exemplo, pode ser necessário enriquecer features de despachos ou revisar exemplos curtos. Se `1` e `3` se confundem, talvez seja preciso modelar melhor trechos com referências a ARE e RE.


In [13]:
best_name, best_holdout_model = select_best_model(results_df, fitted_models, metric="f1_macro")
print("Melhor modelo clássico no holdout:", best_name)

best_info = fitted_models[best_name]
plot_confusion_matrix(best_info["y_valid"], best_info["y_pred"], labels=[0, 1, 2, 3, 4])
display(classification_report_frame(best_info["classification_report"]))


---

## 8. Auditoria de rótulos, pseudo-rotulagem e treino final clássico

A primeira rodada de modelos clássicos serve para escolher uma boa família de baseline. Antes do treino final, usamos esse conhecimento para melhorar a base de treinamento de forma conservadora.

A estratégia tem duas partes independentes:

1. **Auditoria de rótulos existentes:** um modelo probabilístico faz previsões fora da dobra nas amostras rotuladas. Quando a previsão discorda do rótulo original com confiança alta, a linha é registrada em `outputs/potential_label_issues.csv`. Apenas casos com confiança ainda maior são corrigidos automaticamente, mantendo `OriginalCategory` para rastreabilidade.

2. **Pseudo-rotulagem das amostras `Category = -1`:** treinamos o modelo de auditoria nas amostras válidas e aplicamos nas linhas sem rótulo. Só entram no treino final os exemplos com confiança acima do limiar definido. O arquivo `outputs/pseudo_labeled_samples.csv` guarda os pseudo-rótulos e suas confianças.

Com isso, o treino final deixa de ignorar milhares de textos sem rótulo, mas evita contaminar o pipeline com previsões fracas. Se no futuro essa etapa precisar ser refeita, basta ajustar os limiares ou substituir o modelo de auditoria sem mexer no restante do fluxo.


In [14]:
audit_models = build_sparse_model_zoo(random_state=experiment_config.random_state)
label_audit_model_name = "logreg_word_tfidf" if "logreg_word_tfidf" in audit_models else best_name

print("Modelo usado para auditoria e pseudo-rotulagem:", label_audit_model_name)

label_issues = detect_label_issues_with_cv(
    train_clean,
    audit_models[label_audit_model_name],
    config=experiment_config,
    min_confidence=0.85,
)
label_issues_path = OUTPUT_DIR / "potential_label_issues.csv"
label_issues.to_csv(label_issues_path, index=False)

label_audit_model = fit_model_on_full_train(
    train_clean,
    audit_models[label_audit_model_name],
    config=experiment_config,
)

unlabeled_clean = add_clean_text_column(train_unlabeled, config=preprocess_config)
pseudo_labeled, pseudo_report = pseudo_label_unlabeled_samples(
    unlabeled_clean,
    label_audit_model,
    config=experiment_config,
    min_confidence=0.92,
)
pseudo_labeled_path = OUTPUT_DIR / "pseudo_labeled_samples.csv"
pseudo_labeled.to_csv(pseudo_labeled_path, index=False)

train_modeling = build_training_set_with_pseudo_labels(
    train_clean,
    pseudo_labeled,
    label_issues=label_issues,
    config=experiment_config,
    correction_min_confidence=0.97,
)

summary_audit = pd.DataFrame(
    [
        {
            "rotuladas_originais": len(train_clean),
            "sem_rotulo_disponiveis": len(train_unlabeled),
            "pseudo_rotuladas_usadas": pseudo_report["selected"],
            "possiveis_rotulos_ruidosos": len(label_issues),
            "correcoes_automaticas": int((train_modeling["LabelAuditAction"] == "auto_corrected").sum()),
            "total_treino_final": len(train_modeling),
        }
    ]
)
display(summary_audit)
print("Auditoria salva em:", label_issues_path)
print("Pseudo-rótulos salvos em:", pseudo_labeled_path)

if not label_issues.empty:
    display(label_issues.head(10))
if not pseudo_labeled.empty:
    display(pseudo_labeled[["Id", "Category", "PseudoConfidence", "PseudoConfidenceMargin"]].head(10))

final_classical_models = build_sparse_model_zoo(random_state=experiment_config.random_state)
final_classical_model = fit_model_on_full_train(
    train_modeling,
    final_classical_models[best_name],
    config=experiment_config,
)

classical_model_path = save_model(final_classical_model, MODEL_DIR / f"{best_name}.joblib")
submission_classical = generate_submission(
    final_classical_model,
    test_clean,
    output_path=OUTPUT_DIR / "submission.csv",
    text_col="Body_clean",
    id_col="Id",
)

print("Modelo clássico salvo em:", classical_model_path)
print("Submissão clássica salva em:", OUTPUT_DIR / "submission.csv")
display(submission_classical.head())


---

## 9. Representação densa com SentenceTransformers

TF-IDF representa documentos por frequência de termos. Isso é muito forte para textos jurídicos porque termos como `agravo`, `recurso extraordinario`, `acordao`, `sentenca` e fórmulas processuais são altamente discriminativos. Por isso, é normal que um baseline TF-IDF bem ajustado supere embeddings congelados em algumas divisões.

A versão anterior desta seção codificava cada documento inteiro de uma vez. Esse desenho pode derrubar o resultado porque modelos SentenceTransformer têm janela limitada e acabam representando principalmente o início do texto. Para corrigir isso, usamos `build_chunked_sentence_embedding_matrix`: cada documento é dividido em chunks sobrepostos, os chunks são codificados no dispositivo disponível (`cuda`, `mps` ou `cpu`) e depois agregados em um único vetor por documento.

Essa etapa continua sendo independente: ela recebe `Body_clean`, devolve uma matriz densa e treina classificadores em cima dela. Se encontrarmos um embedding jurídico melhor no futuro, só trocamos `embedding_model_name`.


In [15]:
embedding_model_name = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
embedding_batch_size = max(1, runtime_profile["transformer_batch_size"] * 2)

embeddings_train = build_chunked_sentence_embedding_matrix(
    train_modeling["Body_clean"],
    model_name=embedding_model_name,
    batch_size=embedding_batch_size,
    device=runtime_profile["device"],
    max_words=220,
    overlap_words=40,
    pooling="mean",
)
embeddings_test = build_chunked_sentence_embedding_matrix(
    test_clean["Body_clean"],
    model_name=embedding_model_name,
    batch_size=embedding_batch_size,
    device=runtime_profile["device"],
    max_words=220,
    overlap_words=40,
    pooling="mean",
)

embeddings_labeled_only = embeddings_train[: len(train_clean)]
embedding_results, embedding_models = evaluate_embedding_classifier(
    train_clean,
    embeddings_labeled_only,
    config=experiment_config,
)
display(embedding_results)


In [16]:
best_embedding_name = embedding_results.iloc[0]["model"]
final_embedding_model = fit_embedding_classifier_on_full_train(
    train_modeling,
    embeddings_train,
    model_name=best_embedding_name,
    config=experiment_config,
)

embedding_model_path = save_model(final_embedding_model, MODEL_DIR / f"{best_embedding_name}.joblib")
submission_embeddings = generate_embedding_submission(
    final_embedding_model,
    embeddings_test,
    test_clean["Id"],
    output_path=OUTPUT_DIR / "submission_embeddings.csv",
)

print("Modelo de embeddings salvo em:", embedding_model_path)
print("Submissão de embeddings salva em:", OUTPUT_DIR / "submission_embeddings.csv")
display(submission_embeddings.head())


---

## 10. Fine-tuning de Transformer

A etapa final usa um modelo de linguagem pré-treinado. Para português brasileiro, o candidato padrão é o BERTimbau (`neuralmind/bert-base-portuguese-cased`), que tende a ser uma escolha sólida para tarefas em português. O notebook também lista alternativas úteis, como Albertina, Longformer e ModernBERT.

Diferente de uma abordagem baseada apenas em embeddings congelados, o fine-tuning ajusta os pesos do modelo para a tarefa específica. Isso costuma melhorar desempenho quando há dados rotulados suficientes, mas aumenta o custo computacional.

A função `fine_tune_transformer_classifier` ajusta automaticamente:

- dispositivo: `cuda`, `mps` ou `cpu`;
- batch size;
- `fp16`/`bf16` quando CUDA suporta;
- pesos de classe na função de perda, úteis por causa do desbalanceamento;
- compatibilidade com versões novas e antigas do `transformers`, incluindo a mudança de `tokenizer` para `processing_class`;
- salvamento do melhor checkpoint por `f1_macro`.

Em CPU, essa etapa pode demorar bastante, mas o código continua sendo executável. Em Mac com Apple Silicon, o modelo é movido para `mps`. Em servidor NVIDIA, vai para `cuda`.


In [17]:
display(transformer_candidate_models())
print("Ambiente detectado para o fine-tuning:")
display(pd.DataFrame([runtime_profile["environment"]]))


In [ ]:
trainer = fine_tune_transformer_classifier(
    train_modeling,
    model_name=runtime_profile["recommended_transformer"],
    config=experiment_config,
    num_train_epochs=runtime_profile["transformer_epochs"],
    learning_rate=runtime_profile["transformer_learning_rate"],
    per_device_train_batch_size=runtime_profile["transformer_batch_size"],
    max_length=runtime_profile["transformer_max_length"],
    output_dir=str(OUTPUT_DIR / "bertimbau"),
    use_class_weights=True,
    use_lora=runtime_profile["use_lora"],
    quantization_4bit=runtime_profile["quantization_4bit"],
    device=runtime_profile["device"],
)

submission_transformer = generate_transformer_submission(
    trainer,
    test_clean,
    output_path=OUTPUT_DIR / "submission_transformer.csv",
    text_col="Body_clean",
    id_col="Id",
    max_length=runtime_profile["transformer_max_length"],
)

print("Submissão Transformer salva em:", OUTPUT_DIR / "submission_transformer.csv")
display(submission_transformer.head())


---
## 11. Consolidação dos resultados

Ao final do pipeline, teremos pelo menos três submissões:

- `outputs/submission.csv`: melhor modelo clássico;
- `outputs/submission_embeddings.csv`: classificador sobre embeddings densos;
- `outputs/submission_transformer.csv`: modelo Transformer fine-tuned.

A escolha final para envio ao Kaggle deve combinar desempenho local, estabilidade em validação cruzada e resultado público no leaderboard. Em geral, a submissão Transformer tende a ser mais forte, mas os modelos clássicos são úteis como controle: se o Transformer não superar claramente o baseline TF-IDF, isso pode indicar problema de truncamento, hiperparâmetros ou ruído no pré-processamento.


In [ ]:
print("Arquivos gerados em outputs/:")
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", path)

print("Modelos salvos em models/:")
for path in sorted(MODEL_DIR.glob("*.joblib")):
    print("-", path)


---

## 12. Próximos ciclos de melhoria

Com o pipeline completo funcionando, os próximos ciclos devem ser incrementais e independentes:

1. **Pré-processamento:** comparar manter/remover stopwords, testar lematização e avaliar impacto em `f1_macro`;
2. **Auditoria de rótulos:** revisar manualmente `outputs/potential_label_issues.csv` e ajustar os limiares de correção automática;
3. **Pseudo-rotulagem:** testar limiares diferentes e comparar o treino com/sem exemplos pseudo-rotulados;
4. **Textos longos:** comparar pooling por chunks, agregação de probabilidades e modelos de contexto longo;
5. **Modelos clássicos:** ajustar `min_df`, `max_features`, n-gramas e regularização;
6. **Embeddings:** testar modelos jurídicos ou multilíngues mais recentes;
7. **Transformer:** comparar BERTimbau, Albertina e modelos de contexto longo;
8. **PEFT/LoRA:** em ambiente NVIDIA/Linux, instalar `requirements-nvidia.txt` e testar QLoRA para reduzir memória;
9. **Análise de erros:** inspecionar amostras confundidas, principalmente entre ARE e RE.

A estrutura modular permite trocar qualquer uma dessas partes sem reescrever o notebook inteiro.
